## 1. Installation

In [ ]:
# Install required packages
# CRITICAL: Install in this exact order with these exact versions

# Step 1: Core dependencies first
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Step 2: PyAnnote ecosystem (latest compatible versions)
!pip install -q pyannote.core==5.0.0
!pip install -q pyannote.database==5.1.0  
!pip install -q pyannote.metrics==3.2.1
!pip install -q pyannote.audio==3.3.2

# Step 3: Lightning and other dependencies
!pip install -q pytorch-lightning==2.4.0
!pip install -q lightning==2.4.0
!pip install -q tensorboard

# Step 4: Audio processing
!pip install -q pydub librosa soundfile

# Step 5: Utilities
!pip install -q pandas tqdm pyyaml

!pip install "numpy<2.0"

print("✓ All packages installed successfully!")

## 2. Imports and Configuration

In [1]:
import os
import json
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')
print(np.__version__)

# PyTorch
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

# PyAnnote
from pyannote.audio import Model, Inference, Pipeline
from pyannote.audio.tasks import SpeakerDiarization
from pyannote.database import FileFinder, get_protocol, registry
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.diarization import DiarizationErrorRate

# Audio
from pydub import AudioSegment
import torchaudio

# Utils
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

1.26.4
PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4


In [ ]:
# ==================== CONFIGURATION ====================

# Hugging Face Token (required for pre-trained models)
# HF_TOKEN = "lol"  # Replace with your token

# Paths - UPDATE THESE FOR YOUR ENVIRONMENT
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/audio"  # Directory with .wav files
TRAIN_ANNOTATION_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/annotation"  # Directory with .csv/.rttm files
TEST_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio"  # Test audio files

# Output paths
WORK_DIR = "./work"
DATABASE_DIR = f"{WORK_DIR}/database"
MODEL_DIR = f"{WORK_DIR}/models"
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
LOGS_DIR = f"{WORK_DIR}/logs"
SUBMISSION_PATH = f"{WORK_DIR}/submission.csv"

# Create directories
for d in [WORK_DIR, DATABASE_DIR, MODEL_DIR, CHECKPOINT_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# ==================== AUDIO SETTINGS ====================
SAMPLE_RATE = 16000  # Required by pyannote
MONO = True

# ==================== TRAINING SETTINGS ====================
# Model configuration
PRETRAINED_MODEL = "pyannote/segmentation-3.0"  # Best segmentation model
PRETRAINED_EMBEDDING = "pyannote/wespeaker-voxceleb-resnet34-LM"  # Best embedding

# Training hyperparameters
BATCH_SIZE = 32  # Adjust based on GPU memory
NUM_WORKERS = 4
MAX_EPOCHS = 20
LEARNING_RATE = 1e-4
DURATION = 5.0  # seconds per training chunk
MAX_SPEAKERS_PER_CHUNK = 3  # Maximum speakers in a chunk

# Early stopping
PATIENCE = 5
MIN_DELTA = 0.001

# Train/val split
VAL_SPLIT = 0.2

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✓ Configuration loaded")
print(f"✓ Device: {DEVICE}")
print(f"✓ Batch size: {BATCH_SIZE}")
print(f"✓ Max epochs: {MAX_EPOCHS}")

✓ Configuration loaded
✓ Device: cuda
✓ Batch size: 32
✓ Max epochs: 20


## 3. Data Preparation Functions

In [4]:
def convert_timestamp_to_seconds(timestamp_str):
    """
    Convert HH:MM:SS or MM:SS format to seconds.
    """
    if isinstance(timestamp_str, (int, float)):
        return float(timestamp_str)
    
    timestamp_str = str(timestamp_str).strip()
    
    if ':' in timestamp_str:
        parts = timestamp_str.split(':')
        if len(parts) == 3:  # HH:MM:SS
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
        elif len(parts) == 2:  # MM:SS
            m, s = parts
            return int(m) * 60 + float(s)
    
    return float(timestamp_str)


def load_csv_annotation(csv_path):
    """
    Load annotation from CSV file.
    Expected columns: start_time, end_time, speaker_id
    """
    try:
        df = pd.read_csv(csv_path)
        
        # Normalize column names
        df.columns = df.columns.str.lower().str.strip()
        
        # Create annotation
        annotation = Annotation()
        
        for _, row in df.iterrows():
            # Handle different column names
            start_col = [c for c in df.columns if 'start' in c][0]
            end_col = [c for c in df.columns if 'end' in c][0]
            speaker_col = [c for c in df.columns if 'speaker' in c or 'id' in c][0]
            
            start = convert_timestamp_to_seconds(row[start_col])
            end = convert_timestamp_to_seconds(row[end_col])
            speaker_id = row[speaker_col]
            
            # Skip invalid segments
            if pd.isna(start) or pd.isna(end) or pd.isna(speaker_id):
                continue
            if start >= end:
                continue
            
            # Format speaker ID
            speaker = f"speaker_{int(speaker_id):02d}"
            
            # Add segment
            annotation[Segment(start, end)] = speaker
        
        return annotation
    
    except Exception as e:
        print(f"Error loading {csv_path}: {e}")
        return None


def preprocess_audio(input_path, output_path=None):
    """
    Preprocess audio to 16kHz mono WAV.
    """
    if output_path is None:
        output_path = input_path
    
    try:
        audio = AudioSegment.from_wav(input_path)
        
        # Convert to mono
        if audio.channels > 1:
            audio = audio.set_channels(1)
        
        # Resample to 16kHz
        if audio.frame_rate != SAMPLE_RATE:
            audio = audio.set_frame_rate(SAMPLE_RATE)
        
        # Export
        audio.export(output_path, format="wav")
        return output_path
    
    except Exception as e:
        print(f"Error preprocessing {input_path}: {e}")
        return None

print("✓ Data preparation functions loaded")

✓ Data preparation functions loaded


## 4. Create PyAnnote Database

In [5]:
def create_rttm_file(annotation, uri, output_path):
    """
    Create RTTM file from annotation.
    RTTM format: SPEAKER <uri> 1 <start> <duration> <NA> <NA> <speaker> <NA> <NA>
    """
    with open(output_path, 'w') as f:
        for segment, _, speaker in annotation.itertracks(yield_label=True):
            line = f"SPEAKER {uri} 1 {segment.start:.3f} {segment.duration:.3f} <NA> <NA> {speaker} <NA> <NA>\n"
            f.write(line)


def create_uem_file(uri, duration, output_path):
    """
    Create UEM file (defines evaluation boundaries).
    UEM format: <uri> NA <start> <end>
    """
    with open(output_path, 'w') as f:
        f.write(f"{uri} NA 0.000 {duration:.3f}\n")


def prepare_database():
    """
    Prepare PyAnnote database structure with train/validation splits.
    """
    print("\n" + "="*60)
    print("PREPARING DATABASE")
    print("="*60)
    
    # Get all audio files
    audio_files = sorted(Path(TRAIN_AUDIO_DIR).glob("*.wav"))
    print(f"Found {len(audio_files)} audio files")
    
    if len(audio_files) == 0:
        raise ValueError(f"No audio files found in {TRAIN_AUDIO_DIR}")
    
    # Load annotations
    data = []
    for audio_file in tqdm(audio_files, desc="Loading data"):
        uri = audio_file.stem
        
        # Find corresponding annotation
        ann_file = Path(TRAIN_ANNOTATION_DIR) / f"{uri}.csv"
        if not ann_file.exists():
            print(f"Warning: No annotation for {uri}")
            continue
        
        # Load annotation
        annotation = load_csv_annotation(str(ann_file))
        if annotation is None or len(annotation) == 0:
            print(f"Warning: Empty annotation for {uri}")
            continue
        
        # Get audio duration
        waveform, sr = torchaudio.load(str(audio_file))
        duration = waveform.shape[1] / sr
        
        data.append({
            'uri': uri,
            'audio_path': str(audio_file),
            'annotation': annotation,
            'duration': duration
        })
    
    print(f"Loaded {len(data)} valid samples")
    
    # Split into train/val
    train_data, val_data = train_test_split(
        data, 
        test_size=VAL_SPLIT, 
        random_state=42
    )
    
    print(f"Train: {len(train_data)} samples")
    print(f"Validation: {len(val_data)} samples")
    
    # Create database structure
    lists_dir = Path(DATABASE_DIR) / "lists"
    annotations_dir = Path(DATABASE_DIR) / "annotations"
    lists_dir.mkdir(parents=True, exist_ok=True)
    annotations_dir.mkdir(parents=True, exist_ok=True)
    
    def create_subset_files(subset_data, subset_name):
        """Create files for a subset (train or development)."""
        lst_path = lists_dir / f"{subset_name}.lst"
        rttm_path = annotations_dir / f"{subset_name}.rttm"
        uem_path = annotations_dir / f"{subset_name}.uem"
        
        # Create .lst file
        with open(lst_path, 'w') as f:
            for item in subset_data:
                f.write(f"{item['uri']}\n")
        
        # Create .rttm and .uem files
        with open(rttm_path, 'w') as rttm_f, open(uem_path, 'w') as uem_f:
            for item in subset_data:
                uri = item['uri']
                annotation = item['annotation']
                duration = item['duration']
                
                # Write RTTM
                for segment, _, speaker in annotation.itertracks(yield_label=True):
                    line = f"SPEAKER {uri} 1 {segment.start:.3f} {segment.duration:.3f} <NA> <NA> {speaker} <NA> <NA>\n"
                    rttm_f.write(line)
                
                # Write UEM
                uem_f.write(f"{uri} NA 0.000 {duration:.3f}\n")
    
    create_subset_files(train_data, "train")
    create_subset_files(val_data, "development")
    
    # Create database.yml
    database_yml = {
        "Databases": {
            "Custom": f"{TRAIN_AUDIO_DIR}/{{uri}}.wav"
        },
        "Protocols": {
            "Custom": {
                "SpeakerDiarization": {
                    "Training": {
                        "train": {
                            "uri": f"lists/train.lst",
                            "annotation": f"annotations/train.rttm",
                            "annotated": f"annotations/train.uem"
                        },
                        "development": {
                            "uri": f"lists/development.lst",
                            "annotation": f"annotations/development.rttm",
                            "annotated": f"annotations/development.uem"
                        }
                    }
                }
            }
        }
    }
    
    database_yml_path = Path(DATABASE_DIR) / "database.yml"
    with open(database_yml_path, 'w') as f:
        yaml.dump(database_yml, f, default_flow_style=False)
    
    print(f"\n✓ Database created at {DATABASE_DIR}")
    print(f"✓ Database config: {database_yml_path}")
    
    return train_data, val_data, str(database_yml_path)

# Create the database
train_data, val_data, database_yml_path = prepare_database()


PREPARING DATABASE
Found 10 audio files


Loading data:   0%|          | 0/10 [00:00<?, ?it/s]

Loaded 10 valid samples
Train: 8 samples
Validation: 2 samples

✓ Database created at ./work/database
✓ Database config: work/database/database.yml


## 5. Load Protocol

In [6]:
# Load database
registry.load_database(database_yml_path)

# Get protocol
protocol = get_protocol(
    "Custom.SpeakerDiarization.Training",
    preprocessors={"audio": FileFinder()}
)

print("✓ Protocol loaded successfully")

# Verify protocol
train_files = list(protocol.train())
dev_files = list(protocol.development())

print(f"\nProtocol verification:")
print(f"  Train files: {len(train_files)}")
print(f"  Development files: {len(dev_files)}")

if len(train_files) > 0:
    sample = train_files[0]
    print(f"\nSample file:")
    print(f"  URI: {sample['uri']}")
    print(f"  Audio: {sample['audio']}")
    print(f"  Segments: {len(sample['annotation'])}")

'Custom.SpeakerDiarization.Training' found in /kaggle/working/work/database/database.yml does not define the 'scope' of speaker labels (file, database, or global). Setting it to 'file'.
✓ Protocol loaded successfully

Protocol verification:
  Train files: 8
  Development files: 2

Sample file:
  URI: train_006
  Audio: /kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/audio/train_006.wav
  Segments: 348


## 6. Setup Training Task

In [7]:
# Create SpeakerDiarization task
task = SpeakerDiarization(
    protocol,
    duration=DURATION,
    max_speakers_per_chunk=MAX_SPEAKERS_PER_CHUNK,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

print("✓ Training task created")
print(f"  Duration per chunk: {DURATION}s")
print(f"  Max speakers per chunk: {MAX_SPEAKERS_PER_CHUNK}")
print(f"  Batch size: {BATCH_SIZE}")

Protocol Custom.SpeakerDiarization.Training does not precompute the output of torchaudio.info(): adding a 'torchaudio.info' preprocessor for you to speed up dataloaders. See pyannote.database documentation on how to do that yourself.
✓ Training task created
  Duration per chunk: 5.0s
  Max speakers per chunk: 3
  Batch size: 32


## 7. Initialize Model

In [8]:
# Option 1: Train from scratch
model = Model.from_pretrained(
    PRETRAINED_MODEL,
    use_auth_token=HF_TOKEN,
    strict=False  # Allow fine-tuning with different number of speakers
)

model.task = task
# Option 2: Start from a checkpoint (uncomment if resuming)
# model = Model.from_pretrained(
#     "path/to/checkpoint.ckpt",
#     strict=False
# )

print("✓ Model initialized")
print(f"  Base model: {PRETRAINED_MODEL}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/399 [00:00<?, ?B/s]

✓ Model initialized
  Base model: pyannote/segmentation-3.0
  Parameters: 1,473,265


## 8. Setup Training

In [9]:
# Callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="diarization-{epoch:02d}-{val_loss:.4f}",
    monitor="loss/val",
    mode="min",
    save_top_k=3,
    save_last=True,
)

early_stopping = EarlyStopping(
    monitor="loss/val",
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    mode="min",
)

# Logger
logger = TensorBoardLogger(
    save_dir=LOGS_DIR,
    name="diarization_training"
)

# Trainer
trainer = pl.Trainer(
    accelerator="auto",
    devices=1,
    max_epochs=MAX_EPOCHS,
    callbacks=[checkpoint_callback, early_stopping],
    logger=logger,
    gradient_clip_val=1.0,
    log_every_n_steps=10,
)

print("✓ Training setup complete")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  Early stopping patience: {PATIENCE}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Logs: {LOGS_DIR}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


✓ Training setup complete
  Max epochs: 20
  Early stopping patience: 5
  Checkpoints: ./work/checkpoints
  Logs: ./work/logs


## 9. Train Model

In [10]:
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

# Train
trainer.fit(model)

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Best checkpoint: {checkpoint_callback.best_model_path}")
# print(f"Best validation loss: {checkpoint_callback.best_model_score:.4f}")


STARTING TRAINING


2026-02-10 09:54:27.704078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770717267.912264     171 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770717267.967743     171 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770717268.448937     171 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770717268.448976     171 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770717268.448979     171 computation_placer.cc:177] computation placer alr

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


TRAINING COMPLETE
Best checkpoint: /kaggle/working/work/checkpoints/diarization-epoch=08-val_loss=0.0000.ckpt


## 10. Evaluate on Validation Set

## 11. Save Final Model

In [12]:
# Save model
final_model_path = Path(MODEL_DIR) / "final_model"
import shutil
import os

# 1. Get the path to the best checkpoint from your trainer/callback
# (This variable usually exists if you just finished training)
best_ckpt_path = checkpoint_callback.best_model_path

# 2. Define where you want to save your final model
# Pyannote models MUST end in .ckpt to be loaded correctly later
final_path = "work/models/final_model.ckpt"

# Ensure directory exists
os.makedirs(os.path.dirname(final_path), exist_ok=True)

# 3. Copy the file
print(f"Copying best model from: {best_ckpt_path}")
shutil.copy(best_ckpt_path, final_path)

print(f"✓ Model successfully saved to: {final_path}")


# Save pipeline configuration
pipeline_config = {
    "segmentation_model": str(final_path),
    "embedding_model": PRETRAINED_EMBEDDING,
    "parameters": {
    "segmentation": {
        "threshold": 0.5,
        "min_duration_off": 0.0
    },
    "clustering": {
        "method": "centroid",
        "threshold": 0.7,
        "min_cluster_size": 15
    }
}
}

config_path = Path(MODEL_DIR) / "pipeline_config.json"
with open(config_path, 'w') as f:
    json.dump(pipeline_config, f, indent=2)

print(f"✓ Pipeline config saved to {config_path}")

Copying best model from: /kaggle/working/work/checkpoints/diarization-epoch=08-val_loss=0.0000.ckpt
✓ Model successfully saved to: work/models/final_model.ckpt
✓ Pipeline config saved to work/models/pipeline_config.json


## 15. Upload Model to Hugging Face Hub

Upload your trained model to Hugging Face so you can:
- Access it from anywhere
- Share it with others
- Use it in production
- Version control your models

**Before running this cell**:
1. Change `HF_USERNAME` to your Hugging Face username
2. Optionally change `MODEL_NAME` to your desired model name
3. Ensure `HF_TOKEN` is set correctly above

**Note**: This will create a public repository. Set `private=True` if you want it private.

In [ ]:
# ============================================================
# UPLOAD TRAINED MODEL TO HUGGING FACE HUB
# ============================================================
# Add this as a new cell AFTER training completes successfully

# Install required package
!pip install -q huggingface_hub

import os
import json
from pathlib import Path
from huggingface_hub import HfApi, create_repo, upload_folder
from pyannote.audio import Model

print("=" * 70)
print("UPLOADING MODEL TO HUGGING FACE HUB")
print("=" * 70)

# ============================================================
# CONFIGURATION
# ============================================================

# Your Hugging Face credentials
HF_USERNAME = "zarifmahir21"  # ← CHANGE THIS to your HF username
MODEL_NAME = "bengali-speaker-diarization-v1"  # ← CHANGE THIS to your desired model name

# Full repository name
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

# Your HF token (already defined in your notebook)
# HF_TOKEN is already set above

print(f"\nRepository ID: {REPO_ID}")
print(f"Token available: {'✓' if HF_TOKEN else '✗'}")

# ============================================================
# STEP 1: PREPARE MODEL FILES
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: Preparing Model Files")
print("=" * 70)

# Create a directory for HF upload
HF_MODEL_DIR = "./hf_upload"
os.makedirs(HF_MODEL_DIR, exist_ok=True)

# Copy the trained model checkpoint
import shutil

source_model = "work/models/final_model.ckpt"
dest_model = os.path.join(HF_MODEL_DIR, "pytorch_model.bin")

if os.path.exists(source_model):
    print(f"\n✓ Copying model from: {source_model}")
    shutil.copy(source_model, dest_model)
    print(f"✓ Model copied to: {dest_model}")
else:
    print(f"✗ Error: Model not found at {source_model}")
    raise FileNotFoundError(f"Model checkpoint not found at {source_model}")

# ============================================================
# STEP 2: CREATE MODEL CARD (README.md)
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: Creating Model Card")
print("=" * 70)

# Get training metrics if available
try:
    training_der = f"{der*100:.2f}%" if 'der' in locals() else "Not computed"
    num_train_files = len(train_data) if 'train_data' in locals() else "Unknown"
    num_val_files = len(val_data) if 'val_data' in locals() else "Unknown"
    num_epochs = trainer.current_epoch if 'trainer' in locals() else "Unknown"
except:
    training_der = "Not computed"
    num_train_files = "Unknown"
    num_val_files = "Unknown"
    num_epochs = "Unknown"

# Create comprehensive model card
model_card = f"""---
language:
- bn
tags:
- speaker-diarization
- pyannote
- pyannote-audio
- audio
- voice
- speech
- bengali
license: mit
datasets:
- custom
metrics:
- der
model-index:
- name: {MODEL_NAME}
  results:
  - task:
      type: speaker-diarization
      name: Speaker Diarization
    metrics:
    - type: der
      value: {training_der}
      name: Diarization Error Rate
---

# {MODEL_NAME}

This is a fine-tuned speaker diarization model based on pyannote.audio, specifically trained on Bengali audio data.

## Model Description

This model performs speaker diarization - the task of determining "who spoke when" in an audio recording. It can:
- Detect speech segments
- Identify different speakers
- Assign timestamps to each speaker's segments

**Language**: Bengali (বাংলা)

**Base Model**: pyannote/segmentation-3.0

## Training Details

### Training Data
- **Training samples**: {num_train_files}
- **Validation samples**: {num_val_files}
- **Audio format**: 16kHz mono WAV
- **Domain**: Bengali conversational speech

### Training Hyperparameters
- **Epochs**: {num_epochs}
- **Batch size**: {BATCH_SIZE if 'BATCH_SIZE' in globals() else 'Unknown'}
- **Learning rate**: {LEARNING_RATE if 'LEARNING_RATE' in globals() else 'Unknown'}
- **Duration per chunk**: {DURATION if 'DURATION' in globals() else 'Unknown'}s
- **Max speakers per chunk**: {MAX_SPEAKERS_PER_CHUNK if 'MAX_SPEAKERS_PER_CHUNK' in globals() else 'Unknown'}

### Performance
- **Validation DER**: {training_der}

## Usage

### Installation

```bash
pip install pyannote.audio
```

### Basic Usage

```python
from pyannote.audio import Model
from pyannote.audio.pipelines import SpeakerDiarization
import torch

# Load the model
model = Model.from_pretrained("{REPO_ID}")

# Create pipeline
pipeline = SpeakerDiarization(
    segmentation=model,
    embedding="pyannote/wespeaker-voxceleb-resnet34-LM",
    embedding_exclude_overlap=True,
)

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline.to(DEVICE)

# Instantiate with optimal parameters
pipeline.instantiate({{
    "segmentation": {{
        "threshold": 0.5,
    }},
    "clustering": {{
        "method": "centroid",
        "threshold": 0.7,
        "min_cluster_size": 12,
    }},
}})

# Run diarization
diarization = pipeline("audio.wav")

# Print results
for turn, _, speaker in diarization.itertracks(yield_label=True):
    print(f"Speaker {{speaker}}: {{turn.start:.1f}}s - {{turn.end:.1f}}s")
```

### Advanced Usage with GPU Optimization

```python
import torch
from pyannote.audio import Model
from pyannote.audio.pipelines import SpeakerDiarization

# Setup device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # Enable cuDNN autotuner

# Load model
model = Model.from_pretrained("{REPO_ID}")
model.to(DEVICE)
model.eval()

# Create pipeline
pipeline = SpeakerDiarization(
    segmentation=model,
    embedding="pyannote/wespeaker-voxceleb-resnet34-LM",
    embedding_exclude_overlap=True,
)
pipeline.to(DEVICE)

# Instantiate
pipeline.instantiate({{
    "segmentation": {{"threshold": 0.5}},
    "clustering": {{"method": "centroid", "threshold": 0.7}},
}})

# Fast inference
with torch.no_grad():
    diarization = pipeline("audio.wav")

# Get speaker segments
for segment, _, speaker in diarization.itertracks(yield_label=True):
    print(f"[{{segment.start:.2f}}s - {{segment.end:.2f}}s] {{speaker}}")
```

## Model Details

### Architecture
- **Base**: pyannote/segmentation-3.0
- **Type**: PyanNet segmentation model
- **Parameters**: ~1.5M

### Input
- **Format**: WAV audio files
- **Sample rate**: 16kHz
- **Channels**: Mono

### Output
- Speaker diarization annotations (who spoke when)
- Timestamps in seconds
- Speaker labels

## Limitations and Bias

- **Language**: Primarily trained on Bengali audio. Performance may degrade on other languages.
- **Domain**: Optimized for conversational speech. May not perform well on:
  - Very noisy environments
  - Music or singing
  - Overlapping speech (multiple speakers talking simultaneously)
- **Short utterances**: May struggle with very brief speaker turns (<1 second)

## Ethical Considerations

This model is intended for research and development purposes. When deploying in production:
- Ensure compliance with privacy laws and regulations
- Obtain proper consent for audio recording and processing
- Be aware of potential biases in speaker identification
- Do not use for surveillance without proper legal authorization

## Citation

If you use this model, please cite:

```bibtex
@misc{{{MODEL_NAME.replace('-', '_')},
  author = {{{HF_USERNAME}}},
  title = {{Bengali Speaker Diarization Model}},
  year = {{2025}},
  publisher = {{Hugging Face}},
  howpublished = {{\\url{{https://huggingface.co/{REPO_ID}}}}}
}}
```

Also cite the original pyannote.audio paper:

```bibtex
@inproceedings{{Bredin2020,
  Title = {{pyannote.audio: neural building blocks for speaker diarization}},
  Author = {{Herv\\'e Bredin and Ruiqing Yin and Juan Manuel Coria and Gregory Gelly and Pavel Korshunov and Marvin Lavechin and Diego Fustes and Hadrien Titeux and Wassim Bouaziz and Marie-Philippe Gill}},
  Booktitle = {{ICASSP 2020, IEEE International Conference on Acoustics, Speech, and Signal Processing}},
  Year = {{2020}},
}}
```

## License

This model is released under the MIT License.

## Acknowledgements

- Based on [pyannote.audio](https://github.com/pyannote/pyannote-audio)
- Pre-trained on [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0)
- Embedding model: [pyannote/wespeaker-voxceleb-resnet34-LM](https://huggingface.co/pyannote/wespeaker-voxceleb-resnet34-LM)

## Contact

For questions and feedback:
- **Hugging Face**: [{HF_USERNAME}](https://huggingface.co/{HF_USERNAME})
- **Issues**: [GitHub Issues](https://github.com/{HF_USERNAME}/{MODEL_NAME}/issues) (if applicable)
"""

# Save model card
readme_path = os.path.join(HF_MODEL_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(model_card)

print(f"✓ Model card created: {readme_path}")

# ============================================================
# STEP 3: CREATE CONFIG FILES
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: Creating Configuration Files")
print("=" * 70)

# Create config.yaml (pyannote format)
config_yaml = """
# Model configuration for pyannote.audio
task:
  name: SpeakerDiarization
  
architecture:
  name: PyanNet
  
specifications:
  duration: 5.0
  sample_rate: 16000
  
training:
  batch_size: 32
  learning_rate: 0.0001
  max_epochs: 20
"""

config_yaml_path = os.path.join(HF_MODEL_DIR, "config.yaml")
with open(config_yaml_path, "w") as f:
    f.write(config_yaml)

print(f"✓ Config YAML created: {config_yaml_path}")

# Create pipeline config
pipeline_config = {
    "model_type": "speaker-diarization",
    "pyannote_version": "3.3.2",
    "embedding_model": "pyannote/wespeaker-voxceleb-resnet34-LM",
    "optimal_parameters": {
        "segmentation": {
            "threshold": 0.5,
            "min_duration_off": 0.0
        },
        "clustering": {
            "method": "centroid",
            "threshold": 0.7,
            "min_cluster_size": 12
        }
    }
}

pipeline_config_path = os.path.join(HF_MODEL_DIR, "pipeline_config.json")
with open(pipeline_config_path, "w") as f:
    json.dump(pipeline_config, f, indent=2)

print(f"✓ Pipeline config created: {pipeline_config_path}")

# ============================================================
# STEP 4: CREATE USAGE EXAMPLE
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: Creating Usage Example")
print("=" * 70)

usage_example = f"""# Example Usage: {MODEL_NAME}

This example shows how to use the model for speaker diarization.

## Quick Start

```python
from pyannote.audio import Model
from pyannote.audio.pipelines import SpeakerDiarization
import torch

# Load model
model = Model.from_pretrained("{REPO_ID}")

# Create pipeline
pipeline = SpeakerDiarization(
    segmentation=model,
    embedding="pyannote/wespeaker-voxceleb-resnet34-LM",
    embedding_exclude_overlap=True,
)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline.to(device)

# Set parameters
pipeline.instantiate({{
    "segmentation": {{"threshold": 0.5}},
    "clustering": {{"method": "centroid", "threshold": 0.7}},
}})

# Run diarization
diarization = pipeline("your_audio.wav")

# Print results
for turn, _, speaker in diarization.itertracks(yield_label=True):
    print(f"[{{turn.start:.1f}}s - {{turn.end:.1f}}s] {{speaker}}")
```

## Batch Processing

```python
from pathlib import Path
from tqdm import tqdm

audio_files = list(Path("audio_folder").glob("*.wav"))

results = {{}}
with torch.no_grad():
    for audio_file in tqdm(audio_files):
        diarization = pipeline(str(audio_file))
        results[audio_file.stem] = diarization

# Save results
for filename, diarization in results.items():
    with open(f"{{filename}}_diarization.txt", "w") as f:
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            f.write(f"{{turn.start:.2f}} {{turn.end:.2f}} {{speaker}}\\n")
```

## Parameter Tuning

```python
# Adjust segmentation threshold
# Lower = more segments (less missed detection, more false alarms)
# Higher = fewer segments (more missed detection, less false alarms)

pipeline.instantiate({{
    "segmentation": {{"threshold": 0.4}},  # More sensitive
    "clustering": {{"threshold": 0.7}},
}})

# Adjust clustering threshold  
# Lower = more speakers (less confusion, more false speakers)
# Higher = fewer speakers (more confusion, fewer false speakers)

pipeline.instantiate({{
    "segmentation": {{"threshold": 0.5}},
    "clustering": {{"threshold": 0.6}},  # More speakers
}})
```
"""

example_path = os.path.join(HF_MODEL_DIR, "USAGE.md")
with open(example_path, "w") as f:
    f.write(usage_example)

print(f"✓ Usage example created: {example_path}")

# ============================================================
# STEP 5: UPLOAD TO HUGGING FACE HUB
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: Uploading to Hugging Face Hub")
print("=" * 70)

# Verify token
if not HF_TOKEN or HF_TOKEN == "your_token_here":
    print("\n✗ Error: HF_TOKEN not set!")
    print("Please set your Hugging Face token in the configuration section above.")
    raise ValueError("HF_TOKEN not configured")

# Verify username
if HF_USERNAME == "your-username":
    print("\n✗ Error: HF_USERNAME not set!")
    print("Please change HF_USERNAME to your actual Hugging Face username.")
    raise ValueError("HF_USERNAME not configured")

# Initialize API
api = HfApi()

try:
    # Create repository
    print(f"\nCreating repository: {REPO_ID}")
    create_repo(
        repo_id=REPO_ID,
        token=HF_TOKEN,
        private=False,  # Set to True if you want a private repo
        exist_ok=True,  # Don't fail if repo already exists
        repo_type="model"
    )
    print(f"✓ Repository created/verified: https://huggingface.co/{REPO_ID}")
    
    # Upload files
    print(f"\nUploading model files...")
    upload_folder(
        folder_path=HF_MODEL_DIR,
        repo_id=REPO_ID,
        token=HF_TOKEN,
        commit_message="Upload fine-tuned Bengali speaker diarization model",
    )
    
    print(f"\n{'='*70}")
    print("✓ UPLOAD SUCCESSFUL!")
    print(f"{'='*70}")
    print(f"\n🎉 Your model is now available at:")
    print(f"   https://huggingface.co/{REPO_ID}")
    print(f"\n📖 To use your model:")
    print(f"""
from pyannote.audio import Model
model = Model.from_pretrained("{REPO_ID}")
    """)
    
except Exception as e:
    print(f"\n✗ Error during upload: {e}")
    print("\nTroubleshooting:")
    print("1. Verify your HF_TOKEN is correct")
    print("2. Verify your HF_USERNAME is correct")
    print("3. Check your internet connection")
    print("4. Ensure you have write permissions")
    raise

# ============================================================
# STEP 6: CREATE DOWNLOAD INSTRUCTIONS
# ============================================================

print("\n" + "=" * 70)
print("STEP 6: Generating Download Instructions")
print("=" * 70)

download_instructions = f"""# How to Download and Use Your Model

## From Python

```python
from pyannote.audio import Model
from pyannote.audio.pipelines import SpeakerDiarization

# Download and load model (automatic caching)
model = Model.from_pretrained("{REPO_ID}")

# Create pipeline
pipeline = SpeakerDiarization(
    segmentation=model,
    embedding="pyannote/wespeaker-voxceleb-resnet34-LM",
    embedding_exclude_overlap=True,
)

# Instantiate and use
pipeline.instantiate({{
    "segmentation": {{"threshold": 0.5}},
    "clustering": {{"method": "centroid", "threshold": 0.7}},
}})

# Run on audio
diarization = pipeline("audio.wav")
```

## From Command Line

```bash
# Clone repository
git clone https://huggingface.co/{REPO_ID}
cd {MODEL_NAME}

# Use with pyannote
python -c "from pyannote.audio import Model; model = Model.from_pretrained('{REPO_ID}'); print('Model loaded successfully!')"
```

## Model Location

After first download, model is cached at:
- Linux/Mac: `~/.cache/huggingface/hub/models--{HF_USERNAME}--{MODEL_NAME}`
- Windows: `C:\\Users\\<username>\\.cache\\huggingface\\hub\\models--{HF_USERNAME}--{MODEL_NAME}`

## Sharing Your Model

Share this link with others:
https://huggingface.co/{REPO_ID}

## API Access

```python
from huggingface_hub import hf_hub_download

# Download specific file
model_file = hf_hub_download(
    repo_id="{REPO_ID}",
    filename="pytorch_model.bin"
)
```
"""

download_path = os.path.join(HF_MODEL_DIR, "DOWNLOAD.md")
with open(download_path, "w") as f:
    f.write(download_instructions)

print(f"✓ Download instructions created: {download_path}")

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("UPLOAD SUMMARY")
print("=" * 70)

print(f"""
✓ Model repository: {REPO_ID}
✓ URL: https://huggingface.co/{REPO_ID}

Files uploaded:
  - pytorch_model.bin (trained model)
  - README.md (model card)
  - config.yaml (model config)
  - pipeline_config.json (pipeline parameters)
  - USAGE.md (usage examples)
  - DOWNLOAD.md (download instructions)

To use your model anywhere:
  
  from pyannote.audio import Model
  model = Model.from_pretrained("{REPO_ID}")

Share with others:
  https://huggingface.co/{REPO_ID}
""")

print("=" * 70)
